In [1]:
# Download the ZIP file from GitHub
!wget -O BUSBRA.zip "https://github.com/SanchitaMondal/BII_MM60024_Projects/raw/main/data/Breast%20data/BUSBRA.zip"

# Unzip it
!unzip BUSBRA.zip -d /content/data

--2026-04-14 16:04:11--  https://github.com/SanchitaMondal/BII_MM60024_Projects/raw/main/data/Breast%20data/BUSBRA.zip
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/SanchitaMondal/BII_MM60024_Projects/main/data/Breast%20data/BUSBRA.zip [following]
--2026-04-14 16:04:12--  https://raw.githubusercontent.com/SanchitaMondal/BII_MM60024_Projects/main/data/Breast%20data/BUSBRA.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5615303 (5.4M) [application/zip]
Saving to: ‘BUSBRA.zip’

BUSBRA.zip          100%[===================>]   5.35M  --.-KB/s    in 0.06s   

2026-04-14 16:04:12 (89.5 MB/s) - ‘

In [2]:
!ls /content/data/BUSBRA

bus_data.csv  Images  LICENSE.txt  Masks


In [3]:
!pip install torch torchvision pandas scikit-learn opencv-python

In [4]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

In [8]:
DATA_PATH = "/content/data/BUSBRA"

IMG_DIR = os.path.join(DATA_PATH, "Images")
MASK_DIR = os.path.join(DATA_PATH, "Masks")
CSV_PATH = os.path.join(DATA_PATH, "bus_data.csv")

In [12]:
df_original = pd.read_csv(CSV_PATH)

# Create correct filename
df_original['image_name'] = df_original['ID'] + ".png"

# Convert labels
df_original['label'] = df_original['Pathology'].map({
    'benign': 0,
    'malignant': 1
})

# Keep only needed columns
df = df_original[['image_name', 'label']]

df.head()

,image_name,label
0,bus_0001-l.png,1
1,bus_0001-r.png,1
2,bus_0002-l.png,0
3,bus_0002-r.png,0
4,bus_0003-l.png,1


In [13]:
print(df.head())
print("Total samples:", len(df))
print(df['label'].value_counts())

       image_name  label
0  bus_0001-l.png      1
1  bus_0001-r.png      1
2  bus_0002-l.png      0
3  bus_0002-r.png      0
4  bus_0003-l.png      1
Total samples: 106
label
0    56
1    50
Name: count, dtype: int64


In [14]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['label'],
    random_state=42
)

In [15]:
class BreastDataset(Dataset):
    def __init__(self, df, img_dir, mask_dir, use_mask=True):
        self.df = df
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.use_mask = use_mask

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row['image_name']
        label = row['label']

        img_path = os.path.join(self.img_dir, img_name)
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        if image is None:
            raise ValueError(f"Image not found: {img_path}")

        # Resize + normalize
        image = cv2.resize(image, (224, 224))
        image = image / 255.0

        # Apply mask
        if self.use_mask:
            mask_name = img_name.replace("bus", "mask")
            mask_path = os.path.join(self.mask_dir, mask_name)

            if os.path.exists(mask_path):
                mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
                mask = cv2.resize(mask, (224, 224))
                mask = mask / 255.0
                image = image * mask

        # Add channel dimension
        image = np.expand_dims(image, axis=0)

        return torch.tensor(image, dtype=torch.float32), torch.tensor(label, dtype=torch.long)

In [16]:
from torch.utils.data import DataLoader

train_dataset = BreastDataset(train_df, IMG_DIR, MASK_DIR, use_mask=True)
val_dataset = BreastDataset(val_df, IMG_DIR, MASK_DIR, use_mask=True)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)

In [17]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.fc = nn.Sequential(
            nn.Linear(64 * 28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SimpleCNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [19]:
EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss:.4f}")

Epoch 1/10, Loss: 8.0567
Epoch 2/10, Loss: 7.5676
Epoch 3/10, Loss: 7.3985
Epoch 4/10, Loss: 7.1276
Epoch 5/10, Loss: 6.8048
Epoch 6/10, Loss: 6.1531
Epoch 7/10, Loss: 5.9521
Epoch 8/10, Loss: 5.2057
Epoch 9/10, Loss: 4.1026
Epoch 10/10, Loss: 3.1708


In [20]:
model.eval()
y_true, y_pred, y_probs = [], [], []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)

        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)[:, 1]
        preds = torch.argmax(outputs, dim=1)

        y_true.extend(labels.numpy())
        y_pred.extend(preds.cpu().numpy())
        y_probs.extend(probs.cpu().numpy())

from sklearn.metrics import classification_report, roc_auc_score

print(classification_report(y_true, y_pred))
print("AUC:", roc_auc_score(y_true, y_probs))

              precision    recall  f1-score   support

           0       0.73      0.92      0.81        12
           1       0.86      0.60      0.71        10

    accuracy                           0.77        22
   macro avg       0.80      0.76      0.76        22
weighted avg       0.79      0.77      0.77        22

AUC: 0.825


In [30]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['label'],
    random_state=42
)

In [31]:
class BreastDataset(Dataset):
    def __init__(self, df, img_dir, mask_dir, transform=None, use_mask=True):
        self.df = df
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.use_mask = use_mask

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row['image_name']
        label = row['label']

        img_path = os.path.join(self.img_dir, img_name)
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        if image is None:
            raise ValueError(f"Image not found: {img_path}")

        # Resize
        image = cv2.resize(image, (224, 224))

        # Apply mask
        if self.use_mask:
            mask_name = img_name.replace("bus", "mask")
            mask_path = os.path.join(self.mask_dir, mask_name)

            if os.path.exists(mask_path):
                mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
                mask = cv2.resize(mask, (224, 224))
                image = image * (mask / 255.0)

        # Convert to 3 channels (VERY IMPORTANT)
        image = np.stack([image, image, image], axis=-1)

        # Apply transform
        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)

In [32]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor()
])

In [33]:
train_dataset = BreastDataset(train_df, IMG_DIR, MASK_DIR, transform=transform)
val_dataset = BreastDataset(val_df, IMG_DIR, MASK_DIR, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)

In [35]:
from torchvision import models

In [36]:
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(512, 2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [37]:
# Class weights (optional but helpful)
weights = torch.tensor([1.0, 1.5]).to(device)

criterion = nn.CrossEntropyLoss(weight=weights)

# VERY IMPORTANT: lower learning rate
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

In [38]:
EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"[Improved] Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss:.4f}")

[Improved] Epoch 1/10, Loss: 6.5481
[Improved] Epoch 2/10, Loss: 2.8138
[Improved] Epoch 3/10, Loss: 1.7320
[Improved] Epoch 4/10, Loss: 2.8525
[Improved] Epoch 5/10, Loss: 0.9043
[Improved] Epoch 6/10, Loss: 0.8849
[Improved] Epoch 7/10, Loss: 0.3786
[Improved] Epoch 8/10, Loss: 0.6925
[Improved] Epoch 9/10, Loss: 0.3825
[Improved] Epoch 10/10, Loss: 0.1143


In [39]:
model.eval()
y_true, y_pred, y_probs = [], [], []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)

        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)[:, 1]
        preds = torch.argmax(outputs, dim=1)

        y_true.extend(labels.numpy())
        y_pred.extend(preds.cpu().numpy())
        y_probs.extend(probs.cpu().numpy())

print("=== IMPROVED MODEL RESULTS ===")
print(classification_report(y_true, y_pred))
print("AUC:", roc_auc_score(y_true, y_probs))

=== IMPROVED MODEL RESULTS ===
              precision    recall  f1-score   support

           0       0.86      1.00      0.92        12
           1       1.00      0.80      0.89        10

    accuracy                           0.91        22
   macro avg       0.93      0.90      0.91        22
weighted avg       0.92      0.91      0.91        22

AUC: 0.9583333333333334
